In [2]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 75.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=2f4366b9206df998d840c2eb6da9c1ab210a56d39eb7b91ec4f7d02b345524da
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [3]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

# BB84 Quantum Key Distribution Without Attacker

This notebook simulates the BB84 protocol between Alice and Bob only.
There is no eavesdropper, so the error rate should be near 0% and no attack should be detected.

**Protocol outline:**
1. Alice generates random bits and bases, encodes them as qubits
2. Bob measures each qubit using a randomly chosen basis
3. They sift: keep only bits where their bases matched
4. They check a sample of the sifted key for errors
5. If the error rate is below a threshold, the key is accepted

In [5]:
# SHARED UTILITIES

simulator = BasicSimulator()

def quantum_random_bit():
    """
    Generate a single random bit (0 or 1) by placing a qubit in
    superposition |+> = H|0> and measuring it.
    This is a true quantum random number, not a classical PRNG.
    """
    qc = QuantumCircuit(1, 1)
    qc.h(0)          # |0> -> |+> = (|0> + |1>) / sqrt(2)
    qc.measure(0, 0)
    job = simulator.run(transpile(qc, simulator), shots=1)
    result = job.result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

def quantum_random_bits(n):
    """Generate a list of n random bits using quantum measurement."""
    return [quantum_random_bit() for _ in range(n)]

# Number of qubits to send in the protocol
N_QUBITS = 100

# Fraction of sifted key sacrificed for error checking
SAMPLE_FRACTION = 0.2

# Error rate threshold above which an attack is declared
ERROR_THRESHOLD = 0.1  # 10%

print("Utilities ready. Simulator:", simulator.name)

Utilities ready. Simulator: basic_simulator


In [12]:
# ALICE
#Alice randomly selects:
#- A **bit value** (0 or 1) for each qubit
#- A **basis** (0 = rectilinear `+`, 1 = diagonal `×`) for each qubit

def alice_encode(bit, basis):
    """
    Alice encodes a single bit into a qubit circuit.
    basis 0 = rectilinear (+): encode as |0> or |1>
    basis 1 = diagonal   (x): encode as |+> or |->
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)       # |0> -> |1>
    if basis == 1:
        qc.h(0)       # |0> -> |+>  or  |1> -> |->
    return qc

# Alice generates her random bits and bases
print("Alice: generating random bits and bases...")
alice_bits  = quantum_random_bits(N_QUBITS)
alice_bases = quantum_random_bits(N_QUBITS)

# Alice encodes each bit into a qubit circuit (these represent the qubits sent over the channel)
alice_qubits = [alice_encode(alice_bits[i], alice_bases[i]) for i in range(N_QUBITS)]

print(f"Alice bits  (first 20): {alice_bits[:20]}")
print(f"Alice bases (first 20): {alice_bases[:20]}  (0=+, 1=x)")
print(f"\nExample qubit circuit (bit={alice_bits[0]}, basis={alice_bases[0]}):")
print(alice_qubits[0].draw())

Alice: generating random bits and bases...
Alice bits  (first 20): [1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1]
Alice bases (first 20): [1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0]  (0=+, 1=x)

Example qubit circuit (bit=1, basis=1):
     ┌───┐┌───┐
  q: ┤ X ├┤ H ├
     └───┘└───┘
c: 1/══════════
               


In [13]:
#BOB

def bob_measure(qubit_circuit, basis):
    """
    Bob measures a qubit he received from Alice.
    He appends his measurement choice to the circuit.
    basis 0 = rectilinear (+): measure directly
    basis 1 = diagonal   (x): apply H first, then measure
    """
    qc = qubit_circuit.copy()
    if basis == 1:
        qc.h(0)       # rotate back from diagonal basis before measuring
    qc.measure(0, 0)
    job = simulator.run(transpile(qc, simulator), shots=1)
    result = job.result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

# Bob generates his random bases
print("Bob: generating random bases and measuring qubits...")
bob_bases   = quantum_random_bits(N_QUBITS)
bob_results = [bob_measure(alice_qubits[i], bob_bases[i]) for i in range(N_QUBITS)]

print(f"Bob bases   (first 20): {bob_bases[:20]}  (0=+, 1=x)")
print(f"Bob results (first 20): {bob_results[:20]}")

Bob: generating random bases and measuring qubits...
Bob bases   (first 20): [0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]  (0=+, 1=x)
Bob results (first 20): [1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1]


Bob randomly picks a measurement basis for each qubit he receives.
- If he picks the **same basis as Alice**, his result matches Alice's bit.
- If he picks a **different basis**, his result is random (50/50).

To measure in the diagonal basis, Bob applies H before measuring.

In [15]:
# BASIS SIFTING  (classical communication between Alice & Bob)

alice_sifted = []
bob_sifted   = []

for i in range(N_QUBITS):
    if alice_bases[i] == bob_bases[i]:   # bases matched -> keep this bit
        alice_sifted.append(alice_bits[i])
        bob_sifted.append(bob_results[i])

sifted_length = len(alice_sifted)
print(f"Qubits sent       : {N_QUBITS}")
print(f"Sifted key length : {sifted_length}  (~50% expected)")
print(f"\nAlice sifted (first 20): {alice_sifted[:20]}")
print(f"Bob   sifted (first 20): {bob_sifted[:20]}")

Qubits sent       : 100
Sifted key length : 57  (~50% expected)

Alice sifted (first 20): [0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0]
Bob   sifted (first 20): [0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0]


## Error Checking & Attack Detection

Alice and Bob sacrifice a random sample of their sifted key to check for errors.
- In an unattacked channel, errors should be ~0% (any errors are just simulation noise).
- If error rate exceeds the threshold (10%), an attack is declared.

The remaining bits (not used for checking) form the **final secret key**.

In [16]:
# ERROR CHECKING  (classical communication between Alice & Bob)

sample_size = max(1, int(sifted_length * SAMPLE_FRACTION))

# Use quantum randomness to pick which indices to sample
# We pick sample_size indices from the sifted key
sampled_indices = set()
while len(sampled_indices) < sample_size:
    # Generate a random index in range [0, sifted_length)
    # Using enough quantum bits to cover the range
    bits_needed = max(1, math.ceil(math.log2(sifted_length)))
    idx = int("".join(str(quantum_random_bit()) for _ in range(bits_needed)), 2)
    if idx < sifted_length:
        sampled_indices.add(idx)

# Count mismatches in the sample
errors = sum(1 for i in sampled_indices if alice_sifted[i] != bob_sifted[i])
error_rate = errors / sample_size

# Final key = sifted bits NOT used in the sample
final_key_alice = [alice_sifted[i] for i in range(sifted_length) if i not in sampled_indices]
final_key_bob   = [bob_sifted[i]   for i in range(sifted_length) if i not in sampled_indices]

print(f"Sample size : {sample_size} bits ({SAMPLE_FRACTION*100:.0f}% of sifted key)")
print(f"Errors      : {errors}")
print(f"Error rate  : {error_rate*100:.1f}%")
print(f"Threshold   : {ERROR_THRESHOLD*100:.0f}%")
print()

if error_rate > ERROR_THRESHOLD:
    print("ATTACK DETECTED — aborting key exchange!")
else:
    print("No attack detected. Key exchange successful.")
    print(f"\nFinal key length : {len(final_key_alice)} bits")
    print(f"Alice final key  : {final_key_alice}")
    print(f"Bob   final key  : {final_key_bob}")
    keys_match = final_key_alice == final_key_bob
    print(f"Keys match       : {keys_match}")

Sample size : 11 bits (20% of sifted key)
Errors      : 0
Error rate  : 0.0%
Threshold   : 10%

No attack detected. Key exchange successful.

Final key length : 46 bits
Alice final key  : [1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1]
Bob   final key  : [1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1]
Keys match       : True


## Using the Key as a One-Time Pad

Now that Alice and Bob share a secret key, they can use it to actually encrypt and decrypt a message.
A one-time pad works by XOR-ing each bit of the message with the corresponding bit of the key.

- **Encrypt:** `ciphertext = message XOR key`
- **Decrypt:** `message = ciphertext XOR key` (XOR is its own inverse)

As long as the key is secret and never reused, this is **perfectly secure**.

In [17]:
# ONE-TIME PAD ENCRYPTION DEMO
# We can only encrypt as many bits as we have key bits
key_length = len(final_key_alice)
print(f"Available key length: {key_length} bits")

# --- ALICE: creates a secret message (same length as key)
# For demo purposes, generate a random binary message using quantum randomness
message = quantum_random_bits(key_length)
print(f"\nAlice's message  : {message}")
print(f"Alice's key      : {final_key_alice}")

# --- ALICE: encrypts by XOR-ing message with her key
ciphertext = [m ^ k for m, k in zip(message, final_key_alice)]
print(f"\nCiphertext sent  : {ciphertext}")
print("(Alice sends ciphertext to Bob over public channel — safe to intercept, meaningless without key)")

# --- BOB: decrypts by XOR-ing ciphertext with his key
# Bob's key == Alice's key (they agreed on it via BB84)
decrypted = [c ^ k for c, k in zip(ciphertext, final_key_bob)]
print(f"\nBob decrypted    : {decrypted}")

# Verify
success = message == decrypted
print(f"\nDecryption correct: {success}")
if success:
    print("Bob successfully recovered Alice's original message using the shared BB84 key!")

Available key length: 46 bits

Alice's message  : [1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1]
Alice's key      : [1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1]

Ciphertext sent  : [0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0]
(Alice sends ciphertext to Bob over public channel — safe to intercept, meaningless without key)

Bob decrypted    : [1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1]

Decryption correct: True
Bob successfully recovered Alice's original message using the shared BB84 key!
